# duckbox quickstart

Talk to a local duckbox over Arrow Flight SQL and get pandas DataFrames back.

Before this notebook: `docker compose up -d`, seed the lake, and start the daemon (see the README **Quickstart in 5 minutes**). Defaults are `grpc://127.0.0.1:4747` and `policies/ducklake/analyst.yaml`.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if (here / "duckbox_client.py").exists():
    sys.path.insert(0, str(here))
else:
    sys.path.insert(0, str(here / "examples" / "python"))

from duckbox_client import connect, format_error

## Connect

`connect()` sends the tenant key and the bundled sandbox spec on every call. Three lines of user code.

In [ ]:
db = connect(key="tenant_21:user_sergio")
db.key, str(db.spec_path)

## A simple query

Results come back as a pandas DataFrame. The spec rewrites this SQL so Sergio only sees tenant 21's live rows.

In [ ]:
deals = db.query("SELECT id, title, amount FROM deals ORDER BY id")
deals

## Prepared statement

Pass bind values separately. Nothing is interpolated into the SQL string.

In [ ]:
db.query("SELECT id, title, amount FROM deals WHERE id = ?", [1])

## Tenant isolation

Same SQL, different key. Kim must not see Sergio's rows (and deleted rows stay hidden).

In [ ]:
kim = connect(key="tenant_22:user_kim")
kim_deals = kim.query("SELECT id, title, amount FROM deals ORDER BY id")
print("Sergio ids:", list(deals["id"]))
print("Kim ids:   ", list(kim_deals["id"]))
kim_deals

## A rejected query

`secrets` exists in the lake but is not on the spec allowlist, so duckbox refuses it. The helper prints the sandbox reason, not the ADBC wrapper.

In [ ]:
try:
    db.query("SELECT * FROM secrets")
except Exception as exc:
    print(format_error(exc))
db.close()
kim.close()